# 01 — Data Cleaning and Preprocessing

**Project:** Store-Level Retail Demand Forecasting (Rossmann Store Sales)
**Deliverable:** Objective 2, part 1 of 2 — cleaning and preprocessing the raw transactional data before any exploratory analysis or feature engineering happens.

This notebook takes the three raw Kaggle CSVs (`train.csv`, `test.csv`, `store.csv`) and turns `train.csv` into a single, analysis-ready frame: store attributes merged in, categorical fields encoded, missing values imputed with explicit provenance flags (so we never silently pretend a missing value is a real observed value), and a handful of derived exogenous features (competition age, active promotions) computed per row. The cleaned frame is cached to `data/interim/train_merged_clean.parquet` so that notebook 02 (EDA) and notebook 03 (feature engineering — lags, rolling windows) can load it directly instead of re-running this pipeline every time.

All of the actual cleaning logic lives in `src/cleaning.py` and `src/data_io.py`, which are already implemented and unit-tested. This notebook's job is to call that pipeline, verify its output looks correct, and persist it — not to re-implement cleaning logic inline. That separation keeps the pipeline reusable (the exact same `clean_pipeline` will be applied to `test.csv` later, once we get to generating forecasts) and keeps this notebook focused on demonstrating *why* each step is necessary rather than on low-level pandas plumbing.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

## What the cleaning pipeline is about to do

Before running it, here is the sequence of steps `clean_pipeline` performs on `train.csv`, and why each one earns its place rather than being cosmetic:

1. **Merge store attributes onto train.** `store.csv` holds per-store static attributes (`StoreType`, `Assortment`, competition and Promo2 fields) that are not repeated in `train.csv`. These are merged in on `Store` (left join, one row per original train row) so every daily record carries the attributes of the store it belongs to. A row-count assertion inside the pipeline guards against the join silently fanning rows out if `store.csv` ever had duplicate `Store` ids.
2. **Parse and encode `StateHoliday`.** The raw column mixes the string `"0"` (no holiday) with holiday-type codes (`"a"`, `"b"`, `"c"`) and, in some exports, numeric-looking `0.0`/`NaN` artifacts from CSV parsing. These are normalized into a clean `state_holiday_type` category and a binary `is_state_holiday` flag, so downstream code can treat "is it a holiday at all" and "which kind of holiday" as two separate, unambiguous questions.
3. **Impute missing competition and Promo2 fields — with explicit missing flags, not silent guesses.** `CompetitionDistance` is missing for some stores (no competitor has ever been recorded), and `CompetitionOpenSinceMonth/Year` and `Promo2SinceWeek/Year` are missing whenever there simply is no competitor or no Promo2 enrollment for that store. Rather than guessing a plausible-looking fill value and letting the model treat it as if it were an observed fact, the pipeline fills `CompetitionDistance` with the median *and* adds a `competition_distance_missing` flag, and adds a `has_competition_since_info` flag before deriving anything from the since-dates. This way a model (or we, reading the data) can always tell an imputed value apart from a genuinely observed one.
4. **Derive `competition_open_days`.** Using the (possibly-missing, now-flagged) competition since-date, the pipeline computes how many days the nearest competitor has been open as of each row's `Date`, clipped at zero and set to zero wherever we don't actually know a since-date. This turns a sparse categorical "since month/year" pair into a single continuous feature that increases monotonically over time for a given store, which is far more usable by both tree-based and linear models than the raw year/month fields.
5. **Derive `promo2_active`.** `Promo2` is a recurring, cyclical promotion that only runs in specific calendar months per store (given in `PromoInterval`) and only after the store's enrollment date (`Promo2SinceYear`/`Promo2SinceWeek`). The pipeline resolves all of that into one boolean-style flag per row: *is this store currently inside an active Promo2 window on this date*, which is a far more directly useful signal than the three raw columns it's derived from.

All of this happens inside a single `clean_pipeline(df, store)` call so the exact same logic can be reapplied to `test.csv` later without copy-pasting steps.

In [ ]:
import pandas as pd

from src.data_io import read_raw_csvs, save_parquet
from src.cleaning import clean_pipeline

train, test, store = read_raw_csvs("../data/raw")

`read_raw_csvs` loads all three raw files with explicit dtypes (so, for example, `Store` and `DayOfWeek` don't silently end up as `float64`). `test` and `store` are loaded now for convenience and because `clean_pipeline` needs `store` as an argument; `test.csv` itself isn't cleaned or used further in *this* notebook — cleaning and feature engineering for the test/forecast set happens later, reusing the same `clean_pipeline` and `build_feature_matrix` functions, once we actually need to produce forecasts rather than train and validate a model.

Now run the pipeline on `train` and `store`, and check that it did not drop or duplicate any rows. This matters because `train.csv` is one row per (store, date) — losing or duplicating rows here would silently break the per-store date continuity that the lag and rolling-window features in notebook 03 depend on.

In [ ]:
before = len(train)
train_clean = clean_pipeline(train, store)
assert len(train_clean) == before, "clean_pipeline must not drop or duplicate rows"
print(f"Row count preserved: {len(train_clean)}")

## Why `Open == 0` rows are kept, not dropped

It would be tempting to filter out rows where `Open == 0` here, since a closed store trivially has `Sales == 0` and contributes nothing "interesting" to a demand model. We deliberately do **not** do that in this notebook.

The reason is that notebook 03 builds lag features (`lag_1/7/14/28`) and rolling-window features (`roll_mean/std_7/14/28`, `roll_min/max_7`) per store by shifting and windowing a date-indexed series of `Sales`. Those operations assume a **contiguous daily index per store** — day *t* is actually the calendar day immediately before day *t+1* for that store, with no gaps. If we dropped closed days here, a store that is closed every Sunday would have its Saturday and Monday rows sitting next to each other in the frame, and a 7-day lag would silently pull a value from 9 or 10 calendar days ago instead of 7, without raising any error. The rolling means and standard deviations would be computed over the wrong set of days too, biasing them upward (closed days, where `Sales` is legitimately 0, are exactly the low values that keep a demand average honest).

So `Open == 0` rows stay in the cleaned frame, with their true `Sales == 0`. They get filtered out later, after the lag/rolling features have been correctly computed over the full contiguous history and the model-training/evaluation logic specifically needs only open-store rows (that filtering is what `filter_trainable_rows` in `src/features.py` is for, applied downstream of this notebook).

## Sanity check 1: StateHoliday encoding is internally consistent

Before trusting `state_holiday_type` and `is_state_holiday` in any downstream analysis, confirm they agree with each other: every row where `state_holiday_type` is the "no holiday" code (`"0"`) should have `is_state_holiday == 0`, and every row carrying an actual holiday-type code (`"a"`, `"b"`, `"c"`, ...) should have `is_state_holiday == 1`. The cross-tabulation below should therefore show exactly one non-zero combination per holiday type, with no rows where the flag and the type disagree. This also doubles as a first look at how rare state holidays actually are in this data, which matters later when deciding whether `is_state_holiday` is a strong enough signal to use as a standalone model feature.

In [ ]:
train_clean[["state_holiday_type", "is_state_holiday"]].value_counts()

## Sanity check 2: how much of the competition data is actually imputed

`competition_distance_missing` is 1 wherever `CompetitionDistance` was `NaN` and got filled with the median; `has_competition_since_info` is 1 wherever we actually know *when* the nearest competitor opened (as opposed to just knowing one exists, with distance available but no since-date). Taking the mean of each 0/1 flag gives the fraction of rows affected by each kind of imputation. This is the check that justifies step 3 above: if a meaningful fraction of rows have imputed `CompetitionDistance`, it would be misleading to let a model (or a human reading a summary table) treat that column at face value without also seeing these flags.

In [ ]:
train_clean[["competition_distance_missing", "has_competition_since_info"]].mean()

## Sanity check 3: how often Promo2 is actually active

`promo2_active` folds together `Promo2` enrollment, the enrollment since-date, and the store-specific active months from `PromoInterval` into one per-row flag. Looking at its normalized value counts shows what fraction of (store, date) rows fall inside an active Promo2 window versus outside one (either because the store never enrolled in Promo2, or because the date falls outside that store's active months). This is a useful number to sanity-check against intuition: Promo2 is a recurring promotion that is only active roughly a third of the months in a year for an enrolled store, so we'd expect the "active" share to be well under 50%, not close to it.

In [ ]:
train_clean["promo2_active"].value_counts(normalize=True)

## Persisting the cleaned frame

With the row-count assertion and the three sanity checks above all looking correct, cache `train_clean` to `data/interim/train_merged_clean.parquet`. Parquet is used here rather than CSV because it preserves the exact dtypes we just set up (the `state_holiday_type` category, the integer flag columns, the parsed `Date`), reads back faster, and takes less disk space than a CSV round-trip would — all useful since notebook 02 and notebook 03 will each load this file fresh rather than re-running the cleaning pipeline.

In [ ]:
save_parquet(train_clean, "../data/interim/train_merged_clean.parquet")

## Summary

This notebook merged `store.csv` attributes onto `train.csv`, encoded `StateHoliday` into a clean categorical/binary pair, imputed missing competition and Promo2 fields with explicit `*_missing`/`has_*_info` provenance flags rather than silent guesses, and derived `competition_open_days` and `promo2_active` as model-ready exogenous features — all via the single, reusable `clean_pipeline(df, store)` call.

The row-count assertion confirmed the pipeline preserved every one of the original `train.csv` rows (no silent drops or merge-induced duplicates), and `Open == 0` rows were deliberately kept rather than filtered, since dropping them here would break the per-store date continuity that the lag and rolling-window features in notebook 03 depend on.

The result is saved to `data/interim/train_merged_clean.parquet`, with the same row count as the raw `train.csv`, ready to be picked up by notebook 02 (exploratory data analysis) and notebook 03 (feature engineering).